# 10 — Compensate nonloading probes

> **Lesson focus**
>
> **Learn:** remove evidenced probe loads without deleting Port
> identity. **Run:** derive one PTC network view. **Inspect:** selected
> load-state and topology evidence. **Status:** `STABILIZED` Full V1;
> tutorial fixture presentation `CONVERGING` candidate.

## Compensate only declared probes

Lesson 3 used `retain()` to select a boundary without changing effective
topology. This lesson extends the same ordered `ReductionPipeline` with
PTC before `retain()`. The [floating-circuit
fixture](fixtures/floating_probe.py) reconstructs one two-port network
and a floating resonator with probes on both terminals. PTC is an
explicit topology step: it removes only the selected `nonloading_probe`
shunts and preserves their coordinates and wave references. The feedline
is coupled first to a grounded parallel-LC readout. In this lesson that
lumped resonator is the target-mode approximation of a quarter-wave
readout resonator; it is not a claim of full distributed equivalence.
Two distinct coupling capacitors then connect the same `readout_node` to
`floating_plus` and `floating_minus`.

In [ ]:
from fixtures.floating_probe import build_floating_probe_circuit
from scnsim import (
    CircuitRun,
    DirectSolveSpec,
    ReductionPipeline,
    units as u,
)

fixture = build_floating_probe_circuit()
run = CircuitRun(plan=fixture.plan, workspace="workspaces/advanced-course")
ptc_view = run.original.reduce(
    ReductionPipeline()
    .ptc(fixture.probe_plus, fixture.probe_minus)
    .retain("feedline_in", "feedline_out")
)
spec = DirectSolveSpec(frequencies=[5.5, 6.0, 6.5] * u.GHz)

## Inspect the physical chain

The authoring projection shows both readout couplers leaving distinct
taps on the one `readout_node` bus. Those taps are drawing anchors, not
extra circuit nodes. The floating resonator remains an actual mutual C/L
network with its two separate reference shunts and probe boundaries.

In [ ]:
from scnsim import CircuitDiagramSpec, Theme

fixture.plan.render_schematic(
    CircuitDiagramSpec(
        theme=Theme.AUTO,
        show_parameter_values=True,
    )
).show()

## Verify the load evidence

Preflight should name both selected Port loads, their compiled stamps,
and the compensated view. Unknown, terminated, or unevidenced PTC
targets fail closed.

In [ ]:
explanation = run.explain(ptc_view, spec)
explanation.show()

[Previous](09_model_n_trace_line.qmd) · [Course
map](../../docs/index.qmd) · [Next: transform
coordinates](11_transform_retain.qmd) · [Concept: view
lineage](../../docs/concepts/compilation-coordinates-and-network-views.qmd#network-view-lineage)